# Demo F — Pseudo-labels and label spreading

**Class 11 · F1–F2** (slides: `slides/11_autoencoders.md`)

* **F1:** label spreading on a kNN graph (closed form $F = (I - \\alpha S)^{-1} Y_0$): four points by hand, the two moons with two labels, the digits with 50 labels, a sweep of $\\alpha$ and $k$, the twin `LabelSpreading` and a failure case (touching moons).
* **F2:** real-world text: 6000 news headlines (AG News, four topics) as tf-idf vectors, a cosine kNN graph, 8 labels per topic.

In [ ]:
import numpy as np
import polars as pl
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import LabelSpreading
from threadpoolctl import threadpool_limits

from faa import demo, plotlib, semisup
from faa.demo import show, slide

demo.setup("11")
threadpool_limits(1)  # small matrices: one BLAS thread is faster than several
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)

## Slide: Label Spreading by Hand (F1)

Four points on a line, joined in a path ($w_{12} = w_{23} = w_{34} = 1$). Point 1 is labelled class A, point 4 class B, $\\alpha = 0.9$.
$S = D^{-1/2} W D^{-1/2}$ with $D$ the diagonal of the row sums of $W$; scores $F = (I - \\alpha S)^{-1} Y_0$.

In [ ]:
W4 = np.array([[0, 1, 0, 0], [1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0]], dtype=float)
Y0 = np.array([[1, 0], [0, 0], [0, 0], [0, 1]], dtype=float)
S4 = semisup.normalised(W4)
F4 = np.linalg.solve(np.eye(4) - 0.9 * S4, Y0)
slide("Label Spreading by Hand")
print(
    "degrees", W4.sum(1).tolist(), "; S entries", {"s12": round(float(S4[0, 1]), 3), "s23": round(float(S4[1, 2]), 3)}
)
print("scores F (columns A, B):", np.round(F4, 3).tolist())
print("classes:", ["A" if a > b else "B" for a, b in F4])
print("check: F = alpha S F + Y0 for the unnormalised closed form:", bool(np.allclose(F4, 0.9 * S4 @ F4 + Y0)))
Wb = W4.copy()
Wb[1, 2] = Wb[2, 1] = 0.2
Fb = np.linalg.solve(np.eye(4) - 0.9 * semisup.normalised(Wb), Y0)
print("weakening the middle edge to 0.2: scores", np.round(Fb, 3).tolist())

## Slide: A Graph from the Points (F1)

Two moons (300 points, noise 0.08), one label per class. The graph joins each point to its 10 nearest neighbours with a Gaussian weight $w_{ij} = \exp(-\lVert x_i - x_j \rVert^2 / 2\sigma^2)$; $\sigma$ is the median distance to the 10th neighbour.
Label spreading with $\alpha = 0.99$: the labels flow along the graph.

In [ ]:
Xm, ym = make_moons(300, noise=0.08, random_state=1)
lab_m = np.array([np.argmax((ym == 0) & (Xm[:, 0] < -0.8)), np.argmax((ym == 1) & (Xm[:, 0] > 1.8))])
W = semisup.knn_affinity(Xm, 10)
pred = semisup.label_spreading(W, ym, lab_m, 0.99)
slide("A Graph from the Points")
print(
    f"edges {int((W > 0).sum() // 2)}, labelled {lab_m.tolist()}; label spreading accuracy {np.mean(pred == ym):.3f} (line / 1-NN on the 2 labels: 0.703)"
)

fig, ax = plotlib.fig("col", 1.8)
ii, jj = np.nonzero(np.triu(W))
for a, b in zip(ii, jj, strict=True):
    ax.plot(Xm[[a, b], 0], Xm[[a, b], 1], color=GREY, lw=0.25, alpha=0.6)
ax.scatter(Xm[:, 0], Xm[:, 1], s=5, c=np.where(pred == 0, BLUE, ORANGE), zorder=3)
ax.scatter(Xm[lab_m, 0], Xm[lab_m, 1], s=40, c=[BLUE, ORANGE], edgecolor="k", zorder=4)
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
plotlib.save(fig, "f1_graph")

## Slide: Label Spreading on the Digits (F1)

1797 digits as nodes (the test rows join the graph, unlabelled), 50 labels; accuracy on the unlabelled nodes. Sweep of $\alpha$ and of $k$, 5 draws. Twin: scikit-learn `LabelSpreading(kernel="knn")`.

In [ ]:
Xall, yall = semisup.digits_flat()
rows = []
for k in (5, 10, 20):
    Wd = semisup.knn_affinity(Xall, k)
    for alpha in (0.5, 0.9, 0.99):
        accs = []
        for seed in range(5):
            lb = semisup.few_labels(yall, 5, seed)
            p = semisup.label_spreading(Wd, yall, lb, alpha)
            m = np.setdiff1d(np.arange(len(yall)), lb)
            accs.append(np.mean(p[m] == yall[m]))
        rows.append({"k": k, "alpha": alpha, "acc": np.mean(accs), "std": np.std(accs)})
grid = pl.DataFrame(rows)
slide("Label Spreading on the Digits")
show(grid)
lb = semisup.few_labels(yall, 5, 0)
m = np.setdiff1d(np.arange(len(yall)), lb)
ys = yall.copy()
ys[m] = -1
for kern, kw in (("knn", {"n_neighbors": 10}),):
    ls = LabelSpreading(kernel=kern, alpha=0.9, max_iter=200, **kw).fit(Xall, ys)
    print(f"scikit-learn LabelSpreading({kern}): accuracy on unlabelled {np.mean(ls.transduction_[m] == yall[m]):.3f}")
Wd = semisup.knn_affinity(Xall, 10)
ours = semisup.label_spreading(Wd, yall, lb, 0.9)
print(
    f"ours (k=10, alpha=0.9, same 50 labels): accuracy on unlabelled {np.mean(ours[m] == yall[m]):.3f}; self-training on the same 50 labels: see the table above"
)
lr = LogisticRegression(C=10, max_iter=3000).fit(Xall[lb], yall[lb])
print(f"logistic regression on the 50 labels: accuracy on unlabelled {lr.score(Xall[m], yall[m]):.3f}")

fig, ax = plotlib.fig("col", 1.6)
for k, c in zip((5, 10, 20), (BLUE, ORANGE, GREEN), strict=True):
    sub = grid.filter(pl.col("k") == k)
    ax.errorbar(sub["alpha"], sub["acc"], sub["std"], color=c, marker="o", ms=3, capsize=2, label=f"$k = {k}$")
ax.set_xlabel(r"$\alpha$")
ax.set_ylabel("accuracy on unlabelled")
ax.legend(loc="lower right")
plotlib.save(fig, "f1_alpha_k")

## Failure case: a graph that crosses the classes

The same moons with noise 0.3: the two moons touch, neighbours of one class sit across the gap. Spreading loses its advantage.

In [ ]:
rows = []
for noise in (0.05, 0.15, 0.3, 0.5):
    accs, base_acc = [], []
    for seed in range(5):
        Xn, yn = make_moons(300, noise=noise, random_state=seed)
        rng = np.random.default_rng(seed)
        lb = np.concatenate([rng.choice(np.flatnonzero(yn == c), 2, replace=False) for c in (0, 1)])
        W = semisup.knn_affinity(Xn, 10)
        accs.append(np.mean(semisup.label_spreading(W, yn, lb, 0.99) == yn))
        base_acc.append(LogisticRegression(C=10).fit(Xn[lb], yn[lb]).score(Xn, yn))
    rows.append({"noise": noise, "spreading": np.mean(accs), "line_on_4_labels": np.mean(base_acc)})
slide("A graph that crosses the classes")
show(pl.DataFrame(rows))

## Slide: Does It Work on Real Data? News Headlines (F2)

AG News: titles and first sentences of news in four topics (world, sports, business, science/tech). 6000 headlines (1500 per topic) turned into tf-idf vectors (20 000 features, English stop words removed) and joined in a graph of the 10 nearest neighbours by cosine similarity.
Only 8 headlines per topic (32 labels, 0.5 %) are labelled. Accuracy on the other 5968 headlines. Baselines: logistic regression on the 32 labels (ignores the rest) and self-training.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import kneighbors_graph

from faa import data

ag = pl.read_csv(data.data_dir() / "ag_news.csv.zst").filter(pl.col("split") == "train")
sel = pl.concat([ag.filter(pl.col("category") == c).sample(1500, seed=0) for c in sorted(ag["category"].unique())])
texts, topics = sel["text"].to_list(), sel["category"].to_numpy()
yt = np.unique(topics, return_inverse=True)[1]
Xt = TfidfVectorizer(max_features=20000, stop_words="english", sublinear_tf=True).fit_transform(texts)
G = kneighbors_graph(Xt, 10, metric="cosine", mode="connectivity")
Wt = ((G + G.T) > 0).astype(float).tocsr()
slide("Does It Work on Real Data? News Headlines")
print(
    f"{len(texts)} headlines, {Xt.shape[1]} tf-idf features, topics {sorted(set(topics))}; graph with {int(Wt.sum() // 2)} edges (sparse)"
)
rows = []
for per in (2, 4, 8, 16, 32):
    a_ls, a_lr, a_st = [], [], []
    for seed in range(5):
        lb = semisup.few_labels(yt, per, seed)
        m = np.setdiff1d(np.arange(len(yt)), lb)
        p = semisup.label_spreading_sparse(Wt, yt, lb, 0.9, 4)
        a_ls.append(np.mean(p[m] == yt[m]))
        lr_t = LogisticRegression(C=10, max_iter=2000).fit(Xt[lb], yt[lb])
        a_lr.append(lr_t.score(Xt[m], yt[m]))
        st, _ = semisup.self_train(Xt[lb], yt[lb], Xt[m], 0.9, 10)
        a_st.append(st.score(Xt[m], yt[m]))
    rows.append(
        {
            "per_class": per,
            "labels": 4 * per,
            "labelled_only": np.mean(a_lr),
            "self_training": np.mean(a_st),
            "label_spreading": np.mean(a_ls),
            "std_spreading": np.std(a_ls),
        }
    )
news = pl.DataFrame(rows)
show(news)

fig, ax = plotlib.fig("col", 1.7)
ax.plot(news["labels"], news["labelled_only"], color=GREY, marker="o", ms=3, label="labelled only")
ax.plot(news["labels"], news["self_training"], color=BLUE, marker="s", ms=3, label="self-training")
ax.plot(news["labels"], news["label_spreading"], color=ORANGE, marker="^", ms=3, label="label spreading")
ax.set_xscale("log")
ax.set_xlabel("labelled headlines")
ax.set_ylabel("accuracy on the rest")
ax.legend(loc="lower right", fontsize=5.5)
plotlib.save(fig, "f2_news")